In [6]:
import sys
print(sys.executable)

/usr/local/bin/python3.12


In [7]:
%pip install --upgrade pip
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os, sys

print("Current folder:")
print(os.getcwd())

print("\nPython executable:")
print(sys.executable)

print("\nPython path first entries:")
for p in sys.path[:5]:
    print(p)

Current folder:
/Users/massaim/external-projects/testpytorchgpu

Python executable:
/usr/local/bin/python3.12

Python path first entries:
/Users/massaim/external-projects/pytorch-cpu-gpu
/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev
/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/jupyter_debug
/Library/Frameworks/Python.framework/Versions/3.12/lib/python312.zip
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12


In [3]:
import importlib.util

spec = importlib.util.find_spec("numpy")
print(spec)
print(spec.origin)

ModuleSpec(name='numpy', loader=<_frozen_importlib_external.SourceFileLoader object at 0x103ad1730>, origin='/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/__init__.py', submodule_search_locations=['/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy'])
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/__init__.py


In [11]:
%pip uninstall -y numpy
%pip install --no-cache-dir --force-reinstall numpy

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 2.6 MB/s  0:00:02 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os._exit(0)

In [1]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

NumPy: 2.4.6
Torch: 2.12.0
MPS available: True


In [2]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

x = torch.randn(1000, 1000, device=device)
y = x @ x

print(y.device)

Using: mps
mps:0


In [3]:
%pip install scikit-learn pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 2.1 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 2.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip uninstall -y pyarrow

Found existing installation: pyarrow 21.0.0
Uninstalling pyarrow-21.0.0:
  Successfully uninstalled pyarrow-21.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import numpy as np
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
data = fetch_california_housing()

X = data.data
y = data.target.reshape(-1, 1)

print("Features:", data.feature_names)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X shape: (20640, 8)
y shape: (20640, 1)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_test = scaler_y.transform(y_test)

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)

X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test_t = torch.tensor(y_test, dtype=torch.float32, device=device)

Using: mps


In [8]:
n_features = X_train_t.shape[1]

model = torch.nn.Linear(n_features, 1).to(device)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [9]:
epochs = 1000

start = time.time()

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    predictions = model(X_train_t)
    loss = loss_fn(predictions, y_train_t)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.6f}")

end = time.time()

print(f"Training time: {end - start:.2f} seconds")

Epoch 100/1000, Loss: 0.422541
Epoch 200/1000, Loss: 0.390604
Epoch 300/1000, Loss: 0.387565
Epoch 400/1000, Loss: 0.387451
Epoch 500/1000, Loss: 0.387449
Epoch 600/1000, Loss: 0.387449
Epoch 700/1000, Loss: 0.387449
Epoch 800/1000, Loss: 0.387449
Epoch 900/1000, Loss: 0.387449
Epoch 1000/1000, Loss: 0.387449
Training time: 1.17 seconds


In [10]:
epochs = 1000

start = time.time()

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    predictions = model(X_train_t)
    loss = loss_fn(predictions, y_train_t)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.6f}")

end = time.time()

print(f"Training time: {end - start:.2f} seconds")

Epoch 100/1000, Loss: 0.387449
Epoch 200/1000, Loss: 0.387449
Epoch 300/1000, Loss: 0.387449
Epoch 400/1000, Loss: 0.387449
Epoch 500/1000, Loss: 0.387449
Epoch 600/1000, Loss: 0.387449
Epoch 700/1000, Loss: 0.387449
Epoch 800/1000, Loss: 0.387449
Epoch 900/1000, Loss: 0.387449
Epoch 1000/1000, Loss: 0.387449
Training time: 0.29 seconds


In [11]:
with torch.no_grad():
    predictions_scaled = model(X_test_t).cpu().numpy()

predictions_real = scaler_y.inverse_transform(predictions_scaled)
y_test_real = scaler_y.inverse_transform(y_test)

for i in range(10):
    print(
        f"Real: {y_test_real[i][0]:.2f}, "
        f"Predicted: {predictions_real[i][0]:.2f}"
    )

Real: 0.48, Predicted: 0.72
Real: 0.46, Predicted: 1.76
Real: 5.00, Predicted: 2.71
Real: 2.19, Predicted: 2.84
Real: 2.78, Predicted: 2.60
Real: 1.59, Predicted: 2.01
Real: 1.98, Predicted: 2.65
Real: 1.57, Predicted: 2.17
Real: 3.40, Predicted: 2.74
Real: 4.47, Predicted: 3.92


In [12]:
torch.save(model.state_dict(), "linear_regression_mps.pth")